In [1]:
# Working on the Uplift modeling code for the paper "A churn prediction dataset from the telecom sector: a new benchmark for uplift modeling"

In [ ]:
# Churn Prediction and Recommendation System with Uplift Modeling using CasualMl 
# Author: Aryan Atre
# University of Florida - Artificial Intelligence Systems

In [3]:
"""
This notebook implements the complete system as specified in the project PDF:
- Churn prediction using XGBoost/LightGBM
- Uplift modeling with T-learner and S-learner
- Recommendation system with personalized interventions
- SHAP explainability
- Comprehensive evaluation framework

Dataset: Orange Belgium Churn + Uplift Dataset
Source: TheoVerhelst/Churn-Uplift-Dataset-Paper
"""

'\nThis notebook implements the complete system as specified in the project PDF:\n- Churn prediction using XGBoost/LightGBM\n- Uplift modeling with T-learner and S-learner\n- Recommendation system with personalized interventions\n- SHAP explainability\n- Comprehensive evaluation framework\n\nDataset: Orange Belgium Churn + Uplift Dataset\nSource: TheoVerhelst/Churn-Uplift-Dataset-Paper\n'

In [4]:
print("="*80)
print("CHURN PREDICTION AND RECOMMENDATION SYSTEM WITH UPLIFT MODELING")
print("University of Florida - Aryan Atre")
print("="*80)

CHURN PREDICTION AND RECOMMENDATION SYSTEM WITH UPLIFT MODELING
University of Florida - Aryan Atre


In [6]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.ensemble import RandomForestClassifier

# Gradient Boosting 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [7]:
# Uplift Modeling
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "causalml", "-q"])

from causalml.inference.meta import BaseTRegressor, BaseSRegressor
from causalml.metrics import auuc_score, qini_score

In [8]:
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n All libraries imported successfully!")
print(f" Random seed set to: {RANDOM_STATE}")


 All libraries imported successfully!
 Random seed set to: 42


In [9]:
print("\n" + "="*80)
print("SECTION 2: DATA PIPELINE MODULE")
print("="*80)


SECTION 2: DATA PIPELINE MODULE


In [ ]:
# ============================================================================
# SECTION 2: DATA PIPELINE MODULE
# ============================================================================

# 2.1 Data Ingestion
print("\n[2.1] DATA INGESTION")
print("-" * 40)

# Load the Orange Belgium churn dataset
data_path = 'churn_uplift_anonymized.csv'

try:
    df = pd.read_csv(data_path)
    print(f" Dataset loaded successfully!")
    print(f" Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
except FileNotFoundError:
    print(" ERROR: Dataset not found!")
    print(f" Please download from: https://www.dropbox.com/s/27kyinnh9jcjdcg/churn_uplift_anonymized.csv")
    print(f" Save to: {data_path}")
    raise

# 2.2 Data Validation
print("\n[2.2] DATA VALIDATION")
print("-" * 40)

# Schema validation
print("Schema Validation:")
print(f"  ✓ Columns: {df.shape[1]}")
print(f"  ✓ Rows: {df.shape[0]:,}")
print(f"\nColumn Types:")
print(df.dtypes.value_counts())

# Missing values detection
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\n  Missing values detected:")
    print(missing[missing > 0])
else:
    print(f"\n No missing values detected")

# Check for treatment and outcome columns
required_cols = ['treatment', 'churn']
for col in required_cols:
    if col in df.columns:
        print(f"  ✓ '{col}' column found")
    else:
        print(f"   ERROR: '{col}' column missing!")
        raise ValueError(f"Required column '{col}' not found in dataset")

# Display basic statistics
print("\n[2.3] DATA EXPLORATION")
print("-" * 40)

print("\nDataset Preview:")
print(df.head())

print("\nTarget Variable Distribution:")
print(df['churn'].value_counts())
print(f"\nChurn Rate: {df['churn'].mean():.2%}")

print("\nTreatment Distribution:")
print(df['treatment'].value_counts())
print(f"\nTreatment Rate: {df['treatment'].mean():.2%}")

# 2.4 Preprocessing
print("\n[2.4] PREPROCESSING")
print("-" * 40)

# Separate features, treatment, and target
# PDF Section V.C: "Split data into training (60%), validation (20%), and test (20%)"
feature_cols = [col for col in df.columns if col not in ['treatment', 'churn']]
X = df[feature_cols].values
y = df['churn'].values
treatment = df['treatment'].values

print(f"Features: {len(feature_cols)} columns")
print(f"Target: Binary churn (0=retained, 1=churned)")
print(f"Treatment: Binary (0=control, 1=treatment)")

# Train-Validation-Test Split (60-20-20 as per PDF Section VIII.A)
print("\n[2.5] DATA SPLITTING")
print("-" * 40)

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test, t_temp, t_test = train_test_split(
    X, y, treatment,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Second split: 75% train (60% of total), 25% val (20% of total)
X_train, X_val, y_train, y_val, t_train, t_val = train_test_split(
    X_temp, y_temp, t_temp,
    test_size=0.25,  # 0.25 * 0.8 = 0.2 (20% of total)
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]:>6,} samples ({X_train.shape[0]/len(X):.1%})")
print(f"Validation set: {X_val.shape[0]:>6,} samples ({X_val.shape[0]/len(X):.1%})")
print(f"Test set:       {X_test.shape[0]:>6,} samples ({X_test.shape[0]/len(X):.1%})")

# Verify churn distribution in splits
print("\nChurn rate per split:")
print(f"  Train:      {y_train.mean():.2%}")
print(f"  Validation: {y_val.mean():.2%}")
print(f"  Test:       {y_test.mean():.2%}")

# 2.6 Feature Scaling (PDF Section V.C: "Normalize/standardize numeric features")
print("\n[2.6] FEATURE SCALING")
print("-" * 40)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(" Features standardized (mean=0, std=1)")



[2.1] DATA INGESTION
----------------------------------------
 Dataset loaded successfully!
 Shape: 11,896 rows × 180 columns

[2.2] DATA VALIDATION
----------------------------------------
Schema Validation:
  ✓ Columns: 180
  ✓ Rows: 11,896

Column Types:
float64    160
object      18
int64        2
Name: count, dtype: int64

 No missing values detected
   ERROR: 'treatment' column missing!


ValueError: Required column 'treatment' not found in dataset

In [ ]:
# ============================================================================
# SECTION 3: CHURN PREDICTION MODULE
# ============================================================================

print("\n" + "="*80)
print("SECTION 3: CHURN PREDICTION MODULE")
print("="*80)

# 3.1 Baseline Model: Random Forest (PDF Section V.A)
print("\n[3.1] BASELINE MODEL: RANDOM FOREST")
print("-" * 40)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Random Forest (Baseline)...")
rf_model.fit(X_train, y_train)

# Evaluate on validation set
rf_pred_proba = rf_model.predict_proba(X_val)[:, 1]
rf_pred = rf_model.predict(X_val)

rf_auc = roc_auc_score(y_val, rf_pred_proba)
rf_f1 = f1_score(y_val, rf_pred)
rf_precision = precision_score(y_val, rf_pred)
rf_recall = recall_score(y_val, rf_pred)

print(f"\nRandom Forest Performance (Validation Set):")
print(f"  AUC:       {rf_auc:.4f}")
print(f"  F1-Score:  {rf_f1:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")

# 3.2 Primary Model 1: XGBoost (PDF Requirement)
print("\n[3.2] PRIMARY MODEL 1: XGBOOST")
print("-" * 40)

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training XGBoost (Primary Model)...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Evaluate on validation set
xgb_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_pred = xgb_model.predict(X_val)

xgb_auc = roc_auc_score(y_val, xgb_pred_proba)
xgb_f1 = f1_score(y_val, xgb_pred)
xgb_precision = precision_score(y_val, xgb_pred)
xgb_recall = recall_score(y_val, xgb_pred)

print(f"\nXGBoost Performance (Validation Set):")
print(f"  AUC:       {xgb_auc:.4f} {'✅' if xgb_auc >= 0.85 else '⚠️'} (Target: ≥ 0.85)")
print(f"  F1-Score:  {xgb_f1:.4f} {'✅' if xgb_f1 >= 0.75 else '⚠️'} (Target: ≥ 0.75)")
print(f"  Precision: {xgb_precision:.4f}")
print(f"  Recall:    {xgb_recall:.4f}")

# 3.3 Primary Model 2: LightGBM (PDF Requirement)
print("\n[3.3] PRIMARY MODEL 2: LIGHTGBM")
print("-" * 40)

lgb_model = LGBMClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

print("Training LightGBM (Primary Model)...")
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc'
)

# Evaluate on validation set
lgb_pred_proba = lgb_model.predict_proba(X_val)[:, 1]
lgb_pred = lgb_model.predict(X_val)

lgb_auc = roc_auc_score(y_val, lgb_pred_proba)
lgb_f1 = f1_score(y_val, lgb_pred)
lgb_precision = precision_score(y_val, lgb_pred)
lgb_recall = recall_score(y_val, lgb_pred)

print(f"\nLightGBM Performance (Validation Set):")
print(f"  AUC:       {lgb_auc:.4f} {'✅' if lgb_auc >= 0.85 else '⚠️'} (Target: ≥ 0.85)")
print(f"  F1-Score:  {lgb_f1:.4f} {'✅' if lgb_f1 >= 0.75 else '⚠️'} (Target: ≥ 0.75)")
print(f"  Precision: {lgb_precision:.4f}")
print(f"  Recall:    {lgb_recall:.4f}")

# 3.4 Model Comparison
print("\n[3.4] MODEL COMPARISON")
print("-" * 40)

comparison = pd.DataFrame({
    'Model': ['Random Forest (Baseline)', 'XGBoost (Primary)', 'LightGBM (Primary)'],
    'AUC': [rf_auc, xgb_auc, lgb_auc],
    'F1-Score': [rf_f1, xgb_f1, lgb_f1],
    'Precision': [rf_precision, xgb_precision, lgb_precision],
    'Recall': [rf_recall, xgb_recall, lgb_recall],
    'Meets AUC ≥ 0.85': [rf_auc >= 0.85, xgb_auc >= 0.85, lgb_auc >= 0.85],
    'Meets F1 ≥ 0.75': [rf_f1 >= 0.75, xgb_f1 >= 0.75, lgb_f1 >= 0.75]
})

print(comparison.to_string(index=False))

# Select best model
best_idx = comparison['AUC'].idxmax()
best_model_name = comparison.loc[best_idx, 'Model']
best_model = [rf_model, xgb_model, lgb_model][best_idx]

print(f"\n🏆 Best Model Selected: {best_model_name}")
print(f"   AUC: {comparison.loc[best_idx, 'AUC']:.4f}")
print(f"   F1:  {comparison.loc[best_idx, 'F1-Score']:.4f}")

# 3.5 Cross-Validation (PDF Section VIII.A - 5-fold stratified CV)
print("\n[3.5] CROSS-VALIDATION")
print("-" * 40)

print("Running 5-fold Stratified Cross-Validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    best_model, X_train, y_train,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\nCross-Validation AUC Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean CV AUC: {cv_scores.mean():.4f} (± {cv_scores.std():.4f})")

# 3.6 ROC Curve Visualization
print("\n[3.6] ROC CURVE VISUALIZATION")
print("-" * 40)

fig, ax = plt.subplots(figsize=(10, 8))

models_to_plot = [
    (rf_model, rf_pred_proba, 'Random Forest', 'blue'),
    (xgb_model, xgb_pred_proba, 'XGBoost', 'red'),
    (lgb_model, lgb_pred_proba, 'LightGBM', 'green')
]

for model, y_pred_proba, name, color in models_to_plot:
    fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, 
            label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison - Churn Prediction', fontsize=14, fontweight='bold')
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve_comparison.png', dpi=300, bbox_inches='tight')
print(" ROC curve saved to: roc_curve_comparison.png")
plt.show()

# 3.7 Confusion Matrix
print("\n[3.7] CONFUSION MATRIX (BEST MODEL)")
print("-" * 40)

best_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val, best_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
print(" Confusion matrix saved to: confusion_matrix.png")
plt.show()

In [ ]:
# ============================================================================
# SECTION 4: UPLIFT MODELING MODULE
# ============================================================================

print("\n[4.1] T-LEARNER IMPLEMENTATION")
print("-" * 40)

print("Training T-Learner with best churn model as base learner...")
print(f"Base learner: {best_model_name}")

# Use the best model as base learner for uplift
if best_idx == 1:  # XGBoost
    base_learner = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1
    )
elif best_idx == 2:  # LightGBM
    base_learner = LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
else:  # Random Forest
    base_learner = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=-1
    )

# Initialize T-Learner
tlearner = BaseTRegressor(learner=base_learner)

# Fit T-Learner
print("Fitting T-Learner...")
tlearner.fit(X_train, treatment=t_train, y=y_train)

# Predict uplift on validation set
print("Predicting uplift scores...")
uplift_t_val = tlearner.predict(X_val)

print(f"\nT-Learner Uplift Statistics (Validation):")
print(f"  Mean:   {uplift_t_val.mean():.4f}")
print(f"  Std:    {uplift_t_val.std():.4f}")
print(f"  Min:    {uplift_t_val.min():.4f}")
print(f"  Max:    {uplift_t_val.max():.4f}")
print(f"  Median: {np.median(uplift_t_val):.4f}")

# 4.2 S-Learner Implementation (PDF Requirement)
print("\n[4.2] S-LEARNER IMPLEMENTATION")
print("-" * 40)

print("Training S-Learner with best churn model as base learner...")

# Initialize S-Learner (same base learner as T-learner)
if best_idx == 1:
    base_learner_s = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1
    )
elif best_idx == 2:
    base_learner_s = LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
else:
    base_learner_s = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=-1
    )

slearner = BaseSRegressor(learner=base_learner_s)

# Fit S-Learner
print("Fitting S-Learner...")
slearner.fit(X_train, treatment=t_train, y=y_train)

# Predict uplift on validation set
print("Predicting uplift scores...")
uplift_s_val = slearner.predict(X_val)

print(f"\nS-Learner Uplift Statistics (Validation):")
print(f"  Mean:   {uplift_s_val.mean():.4f}")
print(f"  Std:    {uplift_s_val.std():.4f}")
print(f"  Min:    {uplift_s_val.min():.4f}")
print(f"  Max:    {uplift_s_val.max():.4f}")
print(f"  Median: {np.median(uplift_s_val):.4f}")

# 4.3 Uplift Model Evaluation (PDF Section V.D - AUUC, Qini)
print("\n[4.3] UPLIFT MODEL EVALUATION")
print("-" * 40)

# Calculate AUUC (Area Under Uplift Curve)
auuc_t = auuc_score(y_val, uplift_t_val, t_val)
auuc_s = auuc_score(y_val, uplift_s_val, t_val)

# Calculate Qini Coefficient (PDF Section VIII.B.2 - Target > 0.1)
qini_t = qini_score(y_val, uplift_t_val, t_val)
qini_s = qini_score(y_val, uplift_s_val, t_val)

print("Uplift Model Performance:")
print(f"\nT-Learner:")
print(f"  AUUC:           {auuc_t:.4f}")
print(f"  Qini Coeff:     {qini_t:.4f} {'True' if qini_t > 0.1 else 'False'} (Target: > 0.1)")

print(f"\nS-Learner:")
print(f"  AUUC:           {auuc_s:.4f}")
print(f"  Qini Coeff:     {qini_s:.4f} {'True' if qini_s > 0.1 else 'False'} (Target: > 0.1)")

# Select best uplift model
if auuc_t >= auuc_s:
    best_uplift_model = tlearner
    best_uplift_name = "T-Learner"
    best_uplift_scores = uplift_t_val
    best_auuc = auuc_t
    best_qini = qini_t
else:
    best_uplift_model = slearner
    best_uplift_name = "S-Learner"
    best_uplift_scores = uplift_s_val
    best_auuc = auuc_s
    best_qini = qini_s

print(f"\n Best Uplift Model: {best_uplift_name}")
print(f"   AUUC: {best_auuc:.4f}")
print(f"   Qini: {best_qini:.4f}")

# 4.4 Uplift Curve Visualization (PDF Section V.D)
print("\n[4.4] UPLIFT CURVE VISUALIZATION")
print("-" * 40)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# T-Learner uplift curve
plot_uplift(y_val, uplift_t_val, t_val, ax=axes[0])
axes[0].set_title(f'T-Learner Uplift Curve\nAUUC={auuc_t:.4f}, Qini={qini_t:.4f}', 
                  fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# S-Learner uplift curve
plot_uplift(y_val, uplift_s_val, t_val, ax=axes[1])
axes[1].set_title(f'S-Learner Uplift Curve\nAUUC={auuc_s:.4f}, Qini={qini_s:.4f}', 
                  fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('uplift_curves.png', dpi=300, bbox_inches='tight')
print(" Uplift curves saved to: uplift_curves.png")
plt.show()

# 4.5 Segmentation Analysis (PDF Section V.D)
print("\n[4.5] SEGMENTATION ANALYSIS")
print("-" * 40)

# Segment customers by uplift score quartiles
quartiles = np.percentile(best_uplift_scores, [25, 50, 75])
segments = np.digitize(best_uplift_scores, quartiles)

segment_names = ['Low Uplift', 'Medium-Low', 'Medium-High', 'High Uplift']

print("Uplift Segmentation:")
for i, name in enumerate(segment_names):
    mask = (segments == i)
    n_customers = mask.sum()
    avg_uplift = best_uplift_scores[mask].mean()
    churn_rate = y_val[mask].mean()
    treatment_rate = t_val[mask].mean()
    
    print(f"\n{name} (Segment {i+1}):")
    print(f"  Customers:      {n_customers:>6,} ({n_customers/len(y_val):.1%})")
    print(f"  Avg Uplift:     {avg_uplift:>7.4f}")
    print(f"  Churn Rate:     {churn_rate:>7.2%}")
    print(f"  Treatment Rate: {treatment_rate:>7.2%}")



[4.1] T-LEARNER IMPLEMENTATION
----------------------------------------
Training T-Learner with best churn model as base learner...


NameError: name 'best_model_name' is not defined

In [ ]:
# ============================================================================
# SECTION 5: EXPLAINABILITY MODULE
# ============================================================================

# 5.1 SHAP Analysis for Churn Prediction
print("\n[5.1] SHAP ANALYSIS FOR CHURN PREDICTION")
print("-" * 40)

print("Computing SHAP values for churn prediction model...")
print("This may take a few minutes...")

# Create SHAP explainer
if best_idx in [1, 2]:  # XGBoost or LightGBM
    explainer = shap.TreeExplainer(best_model)
else:  # Random Forest
    explainer = shap.TreeExplainer(best_model)

# Calculate SHAP values for validation set (sample for speed)
sample_size = min(500, len(X_val))
sample_indices = np.random.choice(len(X_val), sample_size, replace=False)
X_val_sample = X_val[sample_indices]

shap_values = explainer.shap_values(X_val_sample)

# If binary classification returns list, take values for class 1
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print("SHAP values computed")

# 5.2 SHAP Summary Plot
print("\n[5.2] SHAP SUMMARY PLOT")
print("-" * 40)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values, 
    X_val_sample,
    feature_names=[f"Feature_{i}" for i in range(X_val_sample.shape[1])],
    show=False,
    max_display=15
)
plt.title('SHAP Summary - Top Features Driving Churn', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
print("SHAP summary plot saved to: shap_summary.png")
plt.show()

# 5.3 Feature Importance
print("\n[5.3] FEATURE IMPORTANCE")
print("-" * 40)

# Get feature importance from the model
if hasattr(best_model, 'feature_importances_'):
    feature_importance = best_model.feature_importances_
    
    # Create DataFrame
    importance_df = pd.DataFrame({
        'Feature': [f"Feature_{i}" for i in range(len(feature_importance))],
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False).head(15)
    
    print("\nTop 15 Most Important Features:")
    print(importance_df.to_string(index=False))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(data=importance_df, y='Feature', x='Importance', palette='viridis', ax=ax)
    ax.set_title('Top 15 Feature Importance - Churn Prediction', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    print("\nFeature importance plot saved to: feature_importance.png")
    plt.show()


In [ ]:
# ============================================================================
# SECTION 6: RECOMMENDATION MODULE
# ============================================================================

# 6.1 Generate Recommendations
print("\n[6.1] GENERATING PERSONALIZED RECOMMENDATIONS")
print("-" * 40)

# Get predictions for validation set
churn_proba = best_model.predict_proba(X_val)[:, 1]
uplift_scores = best_uplift_scores

# Create recommendations DataFrame
recommendations = pd.DataFrame({
    'customer_index': range(len(X_val)),
    'churn_probability': churn_proba,
    'uplift_score': uplift_scores,
    'actual_treatment': t_val,
    'actual_churn': y_val
})

# Prioritization logic (PDF: "Prioritize high churn probability AND high uplift")
recommendations['priority_score'] = (
    recommendations['churn_probability'] * 0.6 + 
    recommendations['uplift_score'] * 0.4
)

# Assign intervention (simplified mapping)
def assign_intervention(row):
    if row['churn_probability'] > 0.7 and row['uplift_score'] > 0.1:
        return 'Premium_Support'
    elif row['churn_probability'] > 0.5 and row['uplift_score'] > 0.05:
        return 'Discount_20pct'
    elif row['uplift_score'] > 0.05:
        return 'Personalized_Outreach'
    elif row['churn_probability'] > 0.5:
        return 'Feature_Recommendation'
    else:
        return 'No_Action'

recommendations['recommended_intervention'] = recommendations.apply(assign_intervention, axis=1)

# Sort by priority
recommendations = recommendations.sort_values('priority_score', ascending=False)

print(f"Total customers evaluated: {len(recommendations):,}")
print(f"\nIntervention Distribution:")
print(recommendations['recommended_intervention'].value_counts())

# 6.2 Top Priority Customers
print("\n[6.2] TOP 10 PRIORITY CUSTOMERS FOR INTERVENTION")
print("-" * 40)

top_10 = recommendations.head(10)
display_cols = [
    'customer_index', 'churn_probability', 'uplift_score', 
    'priority_score', 'recommended_intervention'
]
print(top_10[display_cols].to_string(index=False))

# 6.3 Intervention Effectiveness Analysis
print("\n[6.3] INTERVENTION EFFECTIVENESS ANALYSIS")
print("-" * 40)

# Analyze effectiveness by intervention type
intervention_stats = recommendations.groupby('recommended_intervention').agg({
    'churn_probability': ['mean', 'std'],
    'uplift_score': ['mean', 'std'],
    'customer_index': 'count'
}).round(4)

intervention_stats.columns = ['Avg_Churn_Prob', 'Std_Churn_Prob', 
                               'Avg_Uplift', 'Std_Uplift', 'Count']
intervention_stats = intervention_stats.sort_values('Avg_Uplift', ascending=False)

print("\nIntervention Statistics:")
print(intervention_stats)

# 6.4 Recommendation Visualization
print("\n[6.4] RECOMMENDATION VISUALIZATION")
print("-" * 40)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Scatter - Churn Prob vs Uplift
ax = axes[0, 0]
scatter = ax.scatter(
    recommendations['churn_probability'],
    recommendations['uplift_score'],
    c=recommendations['priority_score'],
    cmap='YlOrRd',
    alpha=0.6,
    s=20
)
ax.set_xlabel('Churn Probability', fontsize=11)
ax.set_ylabel('Uplift Score', fontsize=11)
ax.set_title('Customer Segmentation: Churn vs Uplift', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Priority Score')

# Plot 2: Intervention Distribution
ax = axes[0, 1]
intervention_counts = recommendations['recommended_intervention'].value_counts()
ax.bar(range(len(intervention_counts)), intervention_counts.values, color='steelblue')
ax.set_xticks(range(len(intervention_counts)))
ax.set_xticklabels(intervention_counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of Customers', fontsize=11)
ax.set_title('Recommended Intervention Distribution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Priority Score Distribution
ax = axes[1, 0]
ax.hist(recommendations['priority_score'], bins=50, color='teal', alpha=0.7, edgecolor='black')
ax.set_xlabel('Priority Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Priority Scores', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Top Interventions by Avg Uplift
ax = axes[1, 1]
top_interventions = intervention_stats.head(5)
ax.barh(range(len(top_interventions)), top_interventions['Avg_Uplift'].values, color='coral')
ax.set_yticks(range(len(top_interventions)))
ax.set_yticklabels(top_interventions.index)
ax.set_xlabel('Average Uplift Score', fontsize=11)
ax.set_title('Top 5 Interventions by Average Uplift', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('recommendations_analysis.png', dpi=300, bbox_inches='tight')
print(" Recommendation analysis saved to: recommendations_analysis.png")
plt.show()

# 6.5 Save Recommendations to CSV
print("\n[6.5] SAVING RECOMMENDATIONS")
print("-" * 40)

recommendations.to_csv('customer_recommendations.csv', index=False)
print(" Recommendations saved to: customer_recommendations.csv")
print(f"   Total recommendations: {len(recommendations):,}")


In [ ]:
# SECTION 7: FINAL EVALUATION ON TEST SET

# 7.1 Churn Prediction Performance
print("\n[7.1] CHURN PREDICTION - TEST SET PERFORMANCE")
print("-" * 40)

test_churn_pred_proba = best_model.predict_proba(X_test)[:, 1]
test_churn_pred = best_model.predict(X_test)

test_auc = roc_auc_score(y_test, test_churn_pred_proba)
test_f1 = f1_score(y_test, test_churn_pred)
test_precision = precision_score(y_test, test_churn_pred)
test_recall = recall_score(y_test, test_churn_pred)

print(f"Final Test Set Performance ({best_model_name}):")
print(f"  AUC-ROC:   {test_auc:.4f} {'Yes' if test_auc >= 0.85 else 'No'} (Target: ≥ 0.85)")
print(f"  F1-Score:  {test_f1:.4f} {'Yes' if test_f1 >= 0.75 else 'No'} (Target: ≥ 0.75)")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")

# 7.2 Uplift Model Performance
print("\n[7.2] UPLIFT MODELING - TEST SET PERFORMANCE")
print("-" * 40)

test_uplift = best_uplift_model.predict(X_test)

test_auuc = auuc_score(y_test, test_uplift, t_test)
test_qini = qini_score(y_test, test_uplift, t_test)

print(f"Final Test Set Performance ({best_uplift_name}):")
print(f"  AUUC:        {test_auuc:.4f}")
print(f"  Qini Coeff:  {test_qini:.4f} {'Yes' if test_qini > 0.1 else 'No'} (Target: > 0.1)")
print(f"  Mean Uplift: {test_uplift.mean():.4f}")
print(f"  Std Uplift:  {test_uplift.std():.4f}")

# 7.3 Final Summary Report
print("\n[7.3] FINAL PROJECT SUMMARY")
print("-" * 40)

summary = f"""
PROJECT: Churn Prediction and Recommendation System with Uplift Modeling
AUTHOR:  Aryan Atre
INSTITUTION: University of Florida - Artificial Intelligence Systems

DATASET:
  Source: Orange Belgium Churn + Uplift Dataset
  Size:   {len(df):,} customers
  Features: {len(feature_cols)}
  Churn Rate: {df['churn'].mean():.2%}
  Treatment Rate: {df['treatment'].mean():.2%}

DATA SPLIT:
  Training:   {len(X_train):>6,} samples (60%)
  Validation: {len(X_val):>6,} samples (20%)
  Test:       {len(X_test):>6,} samples (20%)

CHURN PREDICTION RESULTS:
  Best Model: {best_model_name}
  
  Validation Performance:
    AUC-ROC:   {comparison.loc[best_idx, 'AUC']:.4f} {'Yes' if comparison.loc[best_idx, 'AUC'] >= 0.85 else 'No'}
    F1-Score:  {comparison.loc[best_idx, 'F1-Score']:.4f} {'Yes' if comparison.loc[best_idx, 'F1-Score'] >= 0.75 else 'No'}
  
  Test Performance:
    AUC-ROC:   {test_auc:.4f} {'Yes' if test_auc >= 0.85 else 'No'}
    F1-Score:  {test_f1:.4f} {'Yes' if test_f1 >= 0.75 else 'No'}

UPLIFT MODELING RESULTS:
  Best Model: {best_uplift_name}
  
  Validation Performance:
    AUUC:      {best_auuc:.4f}
    Qini:      {best_qini:.4f} {'Yes' if best_qini > 0.1 else 'No'}
  
  Test Performance:
    AUUC:      {test_auuc:.4f}
    Qini:      {test_qini:.4f} {'Yes' if test_qini > 0.1 else 'No'}

RECOMMENDATIONS:
  Total Customers: {len(recommendations):,}
  Interventions:
    - Premium Support: {(recommendations['recommended_intervention'] == 'Premium_Support').sum():,}
    - Discount 20%: {(recommendations['recommended_intervention'] == 'Discount_20pct').sum():,}
    - Outreach: {(recommendations['recommended_intervention'] == 'Personalized_Outreach').sum():,}
    - Feature Rec: {(recommendations['recommended_intervention'] == 'Feature_Recommendation').sum():,}
    - No Action: {(recommendations['recommended_intervention'] == 'No_Action').sum():,}

REQUIREMENTS MET:
   AUC ≥ 0.85: {'YES' if test_auc >= 0.85 else 'NO'}
   F1 ≥ 0.75: {'YES' if test_f1 >= 0.75 else 'NO'}
   Qini > 0.1: {'YES' if test_qini > 0.1 else 'NO'}
   XGBoost/LightGBM: YES
   T-learner/S-learner: YES
   SHAP Explainability: YES
   Recommendation System: YES
   Validation Framework: YES

OUTPUTS GENERATED:
  1. roc_curve_comparison.png
  2. confusion_matrix.png
  3. uplift_curves.png
  4. shap_summary.png
  5. feature_importance.png
  6. recommendations_analysis.png
  7. customer_recommendations.csv
"""

print(summary)

# Save summary to file
with open('project_summary.txt', 'w') as f:
    f.write(summary)
print("Project summary saved to: project_summary.txt")

print("\n" + "="*80)
print("PROJECT COMPLETE!")
print("="*80)

print("\nAll modules successfully implemented as per PDF requirements:")
print("   Data Pipeline Module")
print("   Churn Prediction Module (XGBoost/LightGBM)")
print("   Uplift Modeling Module (T-learner/S-learner)")
print("   Explainability Module (SHAP)")
print("   Recommendation Module")
print("   Comprehensive Evaluation Framework")

print("\nNext Steps (as per PDF timeline):")
print("  Week 7-8:  Extend to multi-treatment uplift modeling")
print("  Week 9:    Add recommendation system optimization")
print("  Week 10:   Segmentation analysis")
print("  Week 11:   Monitoring & drift detection")
print("  Week 12:   Final documentation & presentation")